In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [63]:
models ={
    'Oracle-2 Lite': ['../../models/ELAsTiCCv2-lite/smart-river-169/',
                 '../../models/ELAsTiCCv2-lite/playful-waterfall-170/',
                 '../../models/ELAsTiCCv2-lite/happy-glade-175/',
                 '../../models/ELAsTiCCv2-lite/bright-wood-176/',
                 '../../models/ELAsTiCCv2-lite/whole-sun-177/'],
    'Oracle-2': ['../../models/ELAsTiCCv2/icy-violet-168/',
                 '../../models/ELAsTiCCv2/dazzling-river-171/',
                 '../../models/ELAsTiCCv2/rich-serenity-172/',
                 '../../models/ELAsTiCCv2/fine-violet-173/',
                 '../../models/ELAsTiCCv2/wandering-snowball-174/'],

}

colors = {
    'Oracle-2 Lite': '#994882',
    'Oracle-2': '#008080',
    'Oracle-2 Omni': '#FF6645',
    'Image Backbone': "#000000",
}

markers = {
    'Oracle-2 Lite': 's',
    'Oracle-2': 'D',
    'Oracle-2 Omni': 'o',
    'Image Backbone': '',
}

linestyles = {
    'Oracle-2 Lite': 'solid',
    'Oracle-2': 'solid',
    'Oracle-2 Omni': 'solid',
    'Image Backbone': 'dotted',
}

In [64]:
days = 2**np.arange(0, 11)

In [65]:
def get_df(model_path, depth, day):



    path = f"{model_path}/reports/depth{depth}/report_trigger+{day}.csv"
    df = pd.read_csv(path)
    return df
        

In [66]:
def get_tables_for_all_models(model_choice, depth, day):

    tables = []
    cols = None
    names = None

    for model_path in models[model_choice]:
        df = get_df(model_path, depth, day)
        tables.append(df.apply(pd.to_numeric, errors='coerce').to_numpy()[:, 1:])
        cols = df.columns[1:]
        names = df['Class'].to_numpy()

    # find the mean and std for each metric across the 5 runs
    mean_table = np.mean(tables, axis=0)
    std_table = np.std(tables, axis=0)

    # add the columns and rows back from the first table
    new_df = pd.DataFrame()

    new_df['Class'] = names
    for i, col in enumerate(cols):

        # add the mean +/- std as a sring
        new_df[col] = [f"{mean_table[j, i]:.2f}±{std_table[j, i]:.2f}" for j in range(mean_table.shape[0])]

    return new_df
        
    

In [67]:
def get_f1_scores(model_choice, depth, days=[1,8,64, 1024]):

    tables = []

    for d in days:
        df = get_tables_for_all_models(model_choice, depth, d)
        
        # rename column to include the days like f1_d
        df.rename(columns={'f1-score': f'{model_choice} F1_{{{d}}}'}, inplace=True)
        df.drop(columns=['precision', 'recall', 'support'], inplace=True)

        tables.append(df)

    tables = [df.set_index('Class') for df in tables]

    # join the tables on the class
    return pd.concat(tables, axis=1)


In [68]:
def get_model_comparison_f1(depth, days=[1,8,64,1024]):

    tables = []
    for model in models:

        if model=='Image Backbone':
            continue
    
        df = get_f1_scores(model, depth, days)
        tables.append(df)

    # join the tables on the class
    return pd.concat(tables, axis=1)

In [69]:
print(get_model_comparison_f1(1).to_latex())

\begin{tabular}{lllllllll}
\toprule
 & Oracle-2 Lite F1_{1} & Oracle-2 Lite F1_{8} & Oracle-2 Lite F1_{64} & Oracle-2 Lite F1_{1024} & Oracle-2 F1_{1} & Oracle-2 F1_{8} & Oracle-2 F1_{64} & Oracle-2 F1_{1024} \\
Class &  &  &  &  &  &  &  &  \\
\midrule
Transient & 0.98±0.00 & 0.98±0.00 & 1.00±0.00 & 1.00±0.00 & 0.99±0.00 & 0.99±0.00 & 1.00±0.00 & 1.00±0.00 \\
Variable & 0.94±0.00 & 0.95±0.00 & 0.99±0.00 & 1.00±0.00 & 0.96±0.00 & 0.97±0.00 & 1.00±0.00 & 1.00±0.00 \\
accuracy & 0.96±0.00 & 0.97±0.00 & 1.00±0.00 & 1.00±0.00 & 0.98±0.00 & 0.98±0.00 & 1.00±0.00 & 1.00±0.00 \\
macro avg & 0.96±0.00 & 0.97±0.00 & 1.00±0.00 & 1.00±0.00 & 0.97±0.00 & 0.98±0.00 & 1.00±0.00 & 1.00±0.00 \\
weighted avg & 0.97±0.00 & 0.97±0.00 & 1.00±0.00 & 1.00±0.00 & 0.98±0.00 & 0.98±0.00 & 1.00±0.00 & 1.00±0.00 \\
\bottomrule
\end{tabular}



In [70]:
print(get_model_comparison_f1(2).to_latex())

\begin{tabular}{lllllllll}
\toprule
 & Oracle-2 Lite F1_{1} & Oracle-2 Lite F1_{8} & Oracle-2 Lite F1_{64} & Oracle-2 Lite F1_{1024} & Oracle-2 F1_{1} & Oracle-2 F1_{8} & Oracle-2 F1_{64} & Oracle-2 F1_{1024} \\
Class &  &  &  &  &  &  &  &  \\
\midrule
AGN & 0.74±0.01 & 0.81±0.01 & 0.95±0.00 & 0.99±0.00 & 0.93±0.00 & 0.95±0.00 & 0.99±0.00 & 1.00±0.00 \\
Fast & 0.83±0.01 & 0.90±0.00 & 0.97±0.00 & 0.98±0.00 & 0.90±0.00 & 0.94±0.00 & 0.99±0.00 & 0.99±0.00 \\
Long & 0.72±0.00 & 0.76±0.00 & 0.88±0.00 & 0.90±0.00 & 0.80±0.00 & 0.83±0.00 & 0.91±0.00 & 0.93±0.00 \\
Periodic & 0.90±0.00 & 0.93±0.00 & 0.99±0.00 & 1.00±0.00 & 0.97±0.00 & 0.98±0.00 & 1.00±0.00 & 1.00±0.00 \\
SN & 0.69±0.00 & 0.75±0.00 & 0.88±0.00 & 0.90±0.00 & 0.80±0.00 & 0.84±0.00 & 0.92±0.00 & 0.93±0.00 \\
accuracy & 0.77±0.00 & 0.82±0.00 & 0.93±0.00 & 0.94±0.00 & 0.86±0.00 & 0.89±0.00 & 0.95±0.00 & 0.96±0.00 \\
macro avg & 0.78±0.00 & 0.83±0.00 & 0.94±0.00 & 0.95±0.00 & 0.88±0.00 & 0.91±0.00 & 0.96±0.00 & 0.97±0.00 \\
weighted

In [71]:
print(get_model_comparison_f1(-1).to_latex())


\begin{tabular}{lllllllll}
\toprule
 & Oracle-2 Lite F1_{1} & Oracle-2 Lite F1_{8} & Oracle-2 Lite F1_{64} & Oracle-2 Lite F1_{1024} & Oracle-2 F1_{1} & Oracle-2 F1_{8} & Oracle-2 F1_{64} & Oracle-2 F1_{1024} \\
Class &  &  &  &  &  &  &  &  \\
\midrule
AGN & 0.63±0.01 & 0.73±0.01 & 0.95±0.00 & 0.99±0.00 & 0.92±0.00 & 0.94±0.00 & 0.99±0.00 & 1.00±0.00 \\
CART & 0.24±0.01 & 0.31±0.01 & 0.48±0.02 & 0.53±0.02 & 0.38±0.01 & 0.44±0.01 & 0.61±0.01 & 0.65±0.01 \\
Cepheid & 0.79±0.01 & 0.84±0.01 & 0.98±0.00 & 0.99±0.00 & 0.86±0.01 & 0.89±0.01 & 0.99±0.00 & 0.99±0.00 \\
Delta Scuti & 0.60±0.01 & 0.70±0.01 & 0.96±0.00 & 0.99±0.00 & 0.66±0.01 & 0.74±0.01 & 0.97±0.00 & 0.99±0.00 \\
Dwarf Novae & 0.89±0.00 & 0.91±0.00 & 0.96±0.00 & 0.96±0.00 & 0.93±0.00 & 0.95±0.00 & 0.97±0.00 & 0.97±0.00 \\
EB & 0.81±0.00 & 0.86±0.01 & 0.98±0.00 & 0.99±0.00 & 0.86±0.00 & 0.90±0.00 & 0.98±0.00 & 0.99±0.00 \\
ILOT & 0.45±0.00 & 0.49±0.01 & 0.71±0.01 & 0.84±0.00 & 0.60±0.01 & 0.62±0.00 & 0.81±0.01 & 0.88±0.01 \\
KN &